# NB-04: SelfAttention and CrossAttention Modules

## Learning Objectives
- Trace the full SelfAttention forward pass: Q/K are RMSNorm-normalized before RoPE is applied, but V is NOT normalized — understand this asymmetry
- See how the flash_attention wrapper selects backends (flash_attn_3 -> flash_attn_2 -> sage_attn -> PyTorch SDPA) and verify that CPU execution uses the SDPA fallback
- Understand CrossAttention's dual-stream path: when has_image_input=True, the context y is split into CLIP image tokens (y[:, :257]) and T5 text tokens (y[:, 257:])

## Prerequisites
- **Prior notebooks:** NB-01 (RMSNorm), NB-02 (QKV projections, head layout), NB-03 (3D RoPE, rope_apply)
- **Assumed concepts:** Scaled dot-product attention, cross-attention (queries from one source, keys/values from another), residual connections

## Concept Map
- SelfAttention → used in DiT Block for self-attention over video tokens (NB-06)
- CrossAttention → used in DiT Block for text/image conditioning (NB-06)
- flash_attention fallback chain → determines which attention backend runs at inference time (NB-08)

In [ ]:
import sys
import types
import importlib.util
import pathlib

# Find project root: walk up from Course/ to find the directory containing diffsynth/
# This handles both normal checkout and git worktree scenarios.
_here = pathlib.Path().resolve()  # where notebook is executed from
PROJECT_ROOT = None
for _candidate in [_here, _here.parent, _here.parent.parent]:
    if (_candidate / "diffsynth" / "models" / "wan_video_dit.py").exists():
        PROJECT_ROOT = _candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find diffsynth/models/wan_video_dit.py. "
        "Run this notebook from Course/ inside the project checkout."
    )
print(f"Project root: {PROJECT_ROOT}")

# Stub wan_video_camera_controller (not needed for DiT primitive demos)
_cam_stub = types.ModuleType("diffsynth.models.wan_video_camera_controller")
_cam_stub.SimpleAdapter = type("SimpleAdapter", (), {"__init__": lambda s, *a, **kw: None})
_diffsynth_stub = types.ModuleType("diffsynth")
_models_stub = types.ModuleType("diffsynth.models")
sys.modules["diffsynth"] = _diffsynth_stub
sys.modules["diffsynth.models"] = _models_stub
sys.modules["diffsynth.models.wan_video_camera_controller"] = _cam_stub

# Load wan_video_dit.py directly, bypassing the broken diffsynth/__init__.py chain
# Source: importlib.util.spec_from_file_location
_dit_path = PROJECT_ROOT / "diffsynth" / "models" / "wan_video_dit.py"
_spec = importlib.util.spec_from_file_location("diffsynth.models.wan_video_dit", _dit_path)
dit = importlib.util.module_from_spec(_spec)
sys.modules["diffsynth.models.wan_video_dit"] = dit
_spec.loader.exec_module(dit)

# Import the symbols this notebook covers
from diffsynth.models.wan_video_dit import (
    SelfAttention, CrossAttention,
    precompute_freqs_cis_3d, rope_apply, flash_attention,
    RMSNorm, AttentionModule,
)

import torch
import torch.nn as nn
from einops import rearrange

print("Setup complete.")

## 1. SelfAttention Architecture

The `SelfAttention` module is the core self-attention operation used inside each DiT block. It combines four primitives from the earlier notebooks:

1. **Four linear projections (q, k, v, o):** All `dim -> dim`. No dimension expansion — head splitting happens inside the attention backend via einops rearrange.

2. **norm_q and norm_k — RMSNorm applied BEFORE RoPE:** Both `q` and `k` are passed through their respective `RMSNorm` instances before RoPE is applied. This stabilizes the attention dot-product by bounding the magnitude of the query and key vectors.

3. **v is NOT normalized:** The value projection `v = self.v(x)` goes directly into attention without any normalization. This is the key asymmetry. Values carry the actual content to be mixed; normalizing them would change the output magnitude in an undesired way.

4. **RoPE applied after normalization:** `rope_apply(q, freqs, num_heads)` rotates the query and key vectors in the complex plane using precomputed 3D frequencies. Only q and k get RoPE — not v.

**Source:** `diffsynth/models/wan_video_dit.py`, line 125

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 125
B, S, dim, num_heads = 1, 20, 1536, 12
head_dim = dim // num_heads            # 128

sa = SelfAttention(dim=dim, num_heads=num_heads)
print("SelfAttention components:")
for name, module in sa.named_modules():
    if name:
        print(f"  {name}: {module.__class__.__name__}")
total_params = sum(p.numel() for p in sa.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"  q,k,v,o projections: 4 x {dim}x{dim} = {4 * dim * dim:,} weight + {4 * dim:,} bias")
print(f"  norm_q, norm_k: 2 x {dim} = {2 * dim:,}")

## 2. SelfAttention Forward Pass Walkthrough

The forward pass steps for `SelfAttention.forward(x, freqs)`:

```
x  [B, S, dim]
  |-- q = q_proj(x)      [B, S, dim]  (linear projection)
  |   q = norm_q(q)      [B, S, dim]  (RMSNorm: stabilizes dot-product magnitude)
  |   q = rope_apply(q)  [B, S, dim]  (RoPE: encodes 3D position into q)
  |
  |-- k = k_proj(x)      [B, S, dim]  (linear projection)
  |   k = norm_k(k)      [B, S, dim]  (RMSNorm: same reason as q)
  |   k = rope_apply(k)  [B, S, dim]  (RoPE: encodes 3D position into k)
  |
  |-- v = v_proj(x)      [B, S, dim]  (linear projection — NO normalization)
  |
  attn(q, k, v)           [B, S, dim]  (flash_attention SDPA fallback on CPU)
  o = o_proj(attn_out)    [B, S, dim]  (output projection)
```

**3D freqs assembly for this demo:** We build a `[S, 1, head_dim//2]` complex frequency tensor by treating the S positions as a 1D temporal strip (F=S, H=1, W=1). This mirrors the production 3D grid assembly from NB-03 and produces the correct shape for `rope_apply`.

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 141
x = torch.randn(B, S, dim)            # [B, S, dim]

# Build 3D freqs for S positions (F=S, H=1, W=1 — treats sequence as 1D temporal strip)
# See NB-03 for full spatial assembly. The shape rope_apply expects: [S, 1, head_dim//2].
f_freqs, h_freqs, w_freqs = precompute_freqs_cis_3d(head_dim)
F, H, W = S, 1, 1
freqs = torch.cat([
    f_freqs[:F].view(F, 1, 1, -1).expand(F, H, W, -1),  # temporal [F,H,W, f_dim//2]
    h_freqs[:H].view(1, H, 1, -1).expand(F, H, W, -1),  # height   [F,H,W, h_dim//2]
    w_freqs[:W].view(1, 1, W, -1).expand(F, H, W, -1),  # width    [F,H,W, w_dim//2]
], dim=-1).reshape(F * H * W, 1, -1)  # [S, 1, head_dim//2]
assert freqs.shape == torch.Size([S, 1, head_dim // 2]), f"Expected [{S}, 1, {head_dim//2}], got {freqs.shape}"

with torch.no_grad():
    out = sa(x, freqs)                 # [B, S, dim]
assert out.shape == torch.Size([B, S, dim]), f"Expected {(B, S, dim)}, got {out.shape}"
print(f"freqs shape: {freqs.shape}  — [S, 1, head_dim//2] complex per position")
print(f"SelfAttention input:  {x.shape}")
print(f"SelfAttention output: {out.shape}")
print(f"Note: q and k are RMSNorm'd before RoPE; v is NOT normalized")

## 3. Verifying the Q/K Normalization Asymmetry

We can directly inspect the intermediate tensors to confirm that `q` and `k` are normalized while `v` is not. The standard deviation of a normalized vector is typically much lower than an un-normalized projection, since RMSNorm forces the RMS to 1.

In [ ]:
# Demonstrate the asymmetry: q and k are normalized, v is not
with torch.no_grad():
    q_raw = sa.q(x)                    # [B, S, dim] — raw projection
    q_normed = sa.norm_q(q_raw)        # [B, S, dim] — normalized
    k_raw = sa.k(x)                    # [B, S, dim]
    k_normed = sa.norm_k(k_raw)        # [B, S, dim]
    v_raw = sa.v(x)                    # [B, S, dim] — NO normalization applied

print(f"q raw std: {q_raw.std():.4f}, q normed std: {q_normed.std():.4f}")
print(f"k raw std: {k_raw.std():.4f}, k normed std: {k_normed.std():.4f}")
print(f"v raw std: {v_raw.std():.4f} (no normalization)")
print(f"\nAsymmetry: q/k are normalized (bounded magnitude), v is not (carries content scale)")

## 4. CrossAttention Module and the Dual-Stream Path

`CrossAttention` shares the same four-projection structure as `SelfAttention`, but with a crucial difference: **queries come from video tokens (x), while keys and values come from context tokens (y)**.

When `has_image_input=True`, the context `y` is a **concatenated tensor** combining two modalities:
- `img = y[:, :257]` — 1 CLIP CLS token + 256 CLIP patch tokens = 257 total CLIP image tokens
- `ctx = y[:, 257:]` — T5 text encoder tokens (variable length)

The forward pass runs **two separate attention operations**:
1. Main cross-attention: `attn(q, norm_k(k(ctx)), v(ctx))` — video tokens attend to T5 text
2. Image cross-attention: `flash_attention(q, norm_k_img(k_img(img)), v_img(img))` — same query, different keys/values from CLIP
3. The image branch output is **added** to the text branch output before the final output projection.

**Source:** `diffsynth/models/wan_video_dit.py`, line 151

In [ ]:
# Source: diffsynth/models/wan_video_dit.py, line 151
B, S_vid, S_text, dim, num_heads = 1, 20, 30, 1536, 12

# has_image_input=True: y = [CLIP_257_tokens | text_tokens]
ca = CrossAttention(dim=dim, num_heads=num_heads, has_image_input=True)
x = torch.randn(B, S_vid, dim)         # [B, S_vid, dim] — video tokens (queries)
y = torch.randn(B, 257 + S_text, dim)  # [B, 257+S_text, dim] — CLIP(257) + text(S_text)

with torch.no_grad():
    out = ca(x, y)                     # [B, S_vid, dim]
assert out.shape == torch.Size([B, S_vid, dim])
print(f"CrossAttention input x (video):   {x.shape}")
print(f"CrossAttention input y (context): {y.shape}")
print(f"  -> img = y[:, :257]  ({257} CLIP tokens: 1 CLS + 256 patches)")
print(f"  -> ctx = y[:, 257:]  ({S_text} T5 text tokens)")
print(f"CrossAttention output: {out.shape}")

## 5. CrossAttention Without Image Input

When `has_image_input=False`, the context `y` is used directly as the key/value source (no split). The module is simpler: no `k_img`, `v_img`, or `norm_k_img` parameters are created, and only a single attention operation runs.

In [ ]:
ca_text_only = CrossAttention(dim=dim, num_heads=num_heads, has_image_input=False)
y_text = torch.randn(B, S_text, dim)   # [B, S_text, dim] — text only

with torch.no_grad():
    out_text = ca_text_only(x, y_text)  # [B, S_vid, dim]
assert out_text.shape == torch.Size([B, S_vid, dim])
print(f"Text-only CrossAttention output: {out_text.shape}")
# Verify: no k_img/v_img parameters when has_image_input=False
img_params = [n for n, _ in ca_text_only.named_parameters() if 'img' in n]
print(f"Image-related parameters (should be empty): {img_params}")
assert len(img_params) == 0, "has_image_input=False should have no image parameters"

## 6. CrossAttention Parameter Count Comparison

The `has_image_input=True` variant adds three extra parameter groups: `k_img` and `v_img` (each `dim x dim` weight + `dim` bias), and `norm_k_img` (`dim` weight). For `dim=1536`, this image branch adds about 4.7M parameters.

In [ ]:
params_with_img = sum(p.numel() for p in ca.parameters())
params_without_img = sum(p.numel() for p in ca_text_only.parameters())
print(f"CrossAttention params (has_image_input=True):  {params_with_img:,}")
print(f"CrossAttention params (has_image_input=False): {params_without_img:,}")
print(f"Image branch adds: {params_with_img - params_without_img:,} parameters")
print(f"  (k_img: {dim}x{dim} weight + {dim} bias,")
print(f"   v_img: {dim}x{dim} weight + {dim} bias,")
print(f"   norm_k_img: {dim} weight)")

## Summary

### Key Takeaways
- **SelfAttention normalization asymmetry:** `q` and `k` are each passed through `RMSNorm` before RoPE is applied. `v` is NOT normalized. This bounds the dot-product magnitude (stable attention scores) while preserving the value content scale.
- **RoPE after normalization:** RoPE encodes 3D position by rotating the query and key complex vectors. It is applied after RMSNorm — to both `q` and `k`, but not `v`. The freqs tensor must have shape `[S, 1, head_dim//2]`, assembled from all three spatial bands (temporal, height, width).
- **CrossAttention dual-stream path:** When `has_image_input=True`, context `y` is split at index 257: `img = y[:, :257]` (1 CLIP CLS + 256 CLIP patches) and `ctx = y[:, 257:]` (T5 text tokens). Two attention operations run with the same query but different keys/values; results are summed.
- **flash_attention SDPA fallback:** On CPU (no flash_attn_3, flash_attn_2, or sageattention installed), the code falls back to `F.scaled_dot_product_attention` using the head-first layout `[B, N, S, D]`. This is automatically selected — no code change needed.

### Source References
| Symbol | Location |
|--------|-----------|
| SelfAttention | `diffsynth/models/wan_video_dit.py`, line 125 |
| SelfAttention.forward | `diffsynth/models/wan_video_dit.py`, line 141 |
| CrossAttention | `diffsynth/models/wan_video_dit.py`, line 151 |
| flash_attention | `diffsynth/models/wan_video_dit.py`, line 28 |

## Exercises

### Exercise 1 — Disable image branch
Set `has_image_input=False` in the CrossAttention cell. Verify the output shape is unchanged. List all named parameters — confirm `k_img`, `v_img`, and `norm_k_img` are absent.

### Exercise 2 — Change head count
Change `num_heads` from 12 to 6 in SelfAttention. Verify that the output shape is unchanged but `head_dim` doubles to 256. Print the new total parameter count — does it change?

### Exercise 3 — Remove q/k normalization
In your own cell, manually replicate SelfAttention.forward but skip `norm_q` and `norm_k` (pass `self.q(x)` directly to `rope_apply`). Does the output shape change? Compare the output standard deviation to the normalized version.